# 18 - SeamlessM4T v2 Arabic ASR

**Model:** `facebook/seamless-m4t-v2-large`  
**Architecture:** Massively Multilingual Speech-to-Text  
**Supported:** Modern Standard Arabic (arb), Egyptian (arz), Moroccan (ary)

**Tasks:**
1. Dev Set: ASR inference + Diacritization + Metrics
2. Blind Test: ASR inference + Submission

In [1]:
!pip install -q transformers torch accelerate tqdm librosa soundfile torchaudio

In [2]:
# Setup - Import from config.py (generated by setup.sh)
import os, sys, json, re, torch, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
from transformers import AutoProcessor, SeamlessM4Tv2Model, AutoTokenizer, AutoModelForSeq2SeqLM
import librosa
import warnings
warnings.filterwarnings('ignore')

# Add project root to path for config import
IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

# Import paths from config
try:
    from config import (
        PROJECT_DIR, DATA_DIR, DEVICE,
        DEV_AUDIO_DIR, DEV_INPUT, DEV_OUTPUT,
        TEST_DIR, TEST_AUDIO_DIR,
        OUTPUT_DIR, SUBMISSION_DIR
    )
    TEST_INPUT = TEST_DIR / 'test_input.json'
except ImportError:
    print("WARNING: config.py not found. Run setup.sh first!")
    DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
    DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
    DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
    DEV_AUDIO_DIR = DATA_DIR / 'Dev' / 'dev_audio'
    TEST_DIR = PROJECT_DIR / 'Test'
    TEST_INPUT = TEST_DIR / 'test_input.json'
    TEST_AUDIO_DIR = TEST_DIR / 'test_audio'
    OUTPUT_DIR = PROJECT_DIR / 'outputs'
    SUBMISSION_DIR = PROJECT_DIR / 'submissions'

OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True

Environment: Local | Device: cuda
GPU: NVIDIA RTX A5000 (23.6 GB)


In [3]:
MODEL_KEY = 'seamless_m4t'
SEAMLESS_MODEL = 'facebook/seamless-m4t-v2-large'
TASHKEEL_MODEL = 'basharalrfooh/Fine-Tashkeel'

# Load SeamlessM4T for ASR
print(f"Loading {SEAMLESS_MODEL}...")
try:
    seamless_processor = AutoProcessor.from_pretrained(SEAMLESS_MODEL)
    seamless_model = SeamlessM4Tv2Model.from_pretrained(
        SEAMLESS_MODEL,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)
    seamless_model.eval()
    USE_SEAMLESS = True
    print("SeamlessM4T loaded!")
except Exception as e:
    print(f"SeamlessM4T failed: {e}")
    USE_SEAMLESS = False

# Load Fine-Tashkeel for diacritization (ASR output may not be diacritized)
print(f"Loading {TASHKEEL_MODEL}...")
tashkeel_tokenizer = AutoTokenizer.from_pretrained(TASHKEEL_MODEL)
tashkeel_model = AutoModelForSeq2SeqLM.from_pretrained(
    TASHKEEL_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
tashkeel_model.eval()
print("Models loaded!")

Loading facebook/seamless-m4t-v2-large...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

...

SeamlessM4T loaded!
Loading basharalrfooh/Fine-Tashkeel...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/308 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Models loaded!


In [ ]:
def load_audio(audio_path, sr=16000):
    try:
        audio, _ = librosa.load(audio_path, sr=sr)
        return audio
    except Exception as e:
        return None

@torch.inference_mode()
def transcribe_with_seamless(audio):
    """Transcribe audio using SeamlessM4T (outputs Arabic text)."""
    if not USE_SEAMLESS or audio is None:
        return None
    
    try:
        # Process audio (use 'audio' not 'audios' - API changed)
        audio_inputs = seamless_processor(
            audio=audio,
            sampling_rate=16000,
            return_tensors="pt"
        ).to(device)
        
        # Generate transcription (Modern Standard Arabic)
        output_tokens = seamless_model.generate(
            **audio_inputs,
            tgt_lang="arb",
            generate_speech=False
        )
        
        transcription = seamless_processor.decode(
            output_tokens[0].tolist()[0],
            skip_special_tokens=True
        )
        
        return transcription.strip()
    except Exception as e:
        print(f"Seamless error: {e}")
        return None

@torch.inference_mode()
def diacritize_text(text, max_length=1024):
    """Diacritize text using Fine-Tashkeel."""
    inputs = tashkeel_tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    inputs = {k: v.to(tashkeel_model.device) for k, v in inputs.items()}
    outputs = tashkeel_model.generate(**inputs, max_length=max_length, num_beams=4)
    return tashkeel_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

def process_sample(text, audio_path):
    """Process sample using SeamlessM4T ASR, then diacritize the output."""
    # Primary: use SeamlessM4T ASR
    if audio_path and Path(audio_path).exists():
        audio = load_audio(audio_path)
        if audio is not None:
            asr_output = transcribe_with_seamless(audio)
            if asr_output:
                # SeamlessM4T outputs undiacritized text, so diacritize it
                return diacritize_text(asr_output)

    # Fallback to diacritizing input text
    return diacritize_text(text)

In [5]:
# Metrics
DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')
ARABIC_LETTER_PATTERN = re.compile(r'[\u0621-\u064A]')
IRAB_DIACRITICS = {'\u064B', '\u064C', '\u064D'}

def extract_diacritics(text):
    result = []
    i = 0
    while i < len(text):
        if ARABIC_LETTER_PATTERN.match(text[i]):
            diacritics = []
            j = i + 1
            while j < len(text) and DIACRITIC_PATTERN.match(text[j]):
                diacritics.append(text[j])
                j += 1
            result.append((text[i], ''.join(diacritics)))
            i = j
        else:
            i += 1
    return result

def compute_metrics(predictions, ground_truth, exclude_irab=False):
    gt_lookup = {item['id']: item['text_diacritized'] for item in ground_truth}
    total_chars, total_words, char_errors, word_errors, ser_sum, n = 0, 0, 0, 0, 0, 0
    for pred in predictions:
        if pred['id'] not in gt_lookup: continue
        pred_text, ref_text = pred['text_diacritized'], gt_lookup[pred['id']]
        pred_pairs, ref_pairs = extract_diacritics(pred_text), extract_diacritics(ref_text)
        for i in range(min(len(pred_pairs), len(ref_pairs))):
            pred_d, ref_d = pred_pairs[i][1], ref_pairs[i][1]
            if exclude_irab:
                pred_d = ''.join(d for d in pred_d if d not in IRAB_DIACRITICS)
                ref_d = ''.join(d for d in ref_d if d not in IRAB_DIACRITICS)
            if pred_d != ref_d: char_errors += 1
        char_errors += abs(len(pred_pairs) - len(ref_pairs))
        total_chars += max(len(pred_pairs), len(ref_pairs))
        pred_words, ref_words = pred_text.split(), ref_text.split()
        for i in range(min(len(pred_words), len(ref_words))):
            if pred_words[i] != ref_words[i]: word_errors += 1
        word_errors += abs(len(pred_words) - len(ref_words))
        total_words += max(len(pred_words), len(ref_words))
        if pred_text != ref_text: ser_sum += 1
        n += 1
    return {'DER': char_errors/total_chars if total_chars else 0, 'WER': word_errors/total_words if total_words else 0, 'SER': ser_sum/n if n else 0, 'n_samples': n}

## Dev Set Inference

In [ ]:
# Checkpoint support for resume
CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'processed_ids': [p['id'] for p in predictions],
            'predictions': predictions
        }, f, ensure_ascii=False)

with open(DEV_INPUT, 'r', encoding='utf-8') as f: dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f: dev_output = json.load(f)
print(f"Dev samples: {len(dev_input)}")

# Load checkpoint if exists
processed_ids, dev_predictions = load_checkpoint()
print(f"Resuming from checkpoint: {len(processed_ids)} already processed")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids:
        continue
    try:
        audio_path = DEV_AUDIO_DIR / f"{item['id']}.mp3"
        if not audio_path.exists():
            audio_path = DEV_AUDIO_DIR / f"{item['id']}.wav"
        
        diacritized = process_sample(item['text_undiacritized'], audio_path)
        dev_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
        save_checkpoint(dev_predictions)
    except Exception as e:
        print(f"Error {item['id']}: {e}")
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

In [7]:
print("="*60 + "\nDEV SET RESULTS\n" + "="*60)
m1 = compute_metrics(dev_predictions, dev_output, False)
print(f"\n[With I'rab] DER: {m1['DER']*100:.2f}% | WER: {m1['WER']*100:.2f}% (PRIMARY) | SER: {m1['SER']*100:.2f}%")
m2 = compute_metrics(dev_predictions, dev_output, True)
print(f"[No I'rab]   DER: {m2['DER']*100:.2f}% | WER: {m2['WER']*100:.2f}% | SER: {m2['SER']*100:.2f}%")

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

DEV SET RESULTS

[With I'rab] DER: 13.18% | WER: 40.87% (PRIMARY) | SER: 78.46%
[No I'rab]   DER: 13.13% | WER: 40.87% | SER: 78.46%


## Blind Test Inference

In [ ]:
# Checkpoint support for test set
TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'processed_ids': [p['id'] for p in predictions],
            'predictions': predictions
        }, f, ensure_ascii=False)

with open(TEST_INPUT, 'r', encoding='utf-8') as f: test_input = json.load(f)
print(f"Test samples: {len(test_input)}")

# Load checkpoint if exists
test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Resuming from checkpoint: {len(test_processed_ids)} already processed")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids:
        continue
    try:
        audio_path = TEST_AUDIO_DIR / f"{item['id']}.mp3"
        if not audio_path.exists():
            audio_path = TEST_AUDIO_DIR / f"{item['id']}.wav"
        
        diacritized = process_sample(item['text_undiacritized'], audio_path)
        test_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
        save_test_checkpoint(test_predictions)
    except Exception as e:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)
zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print(f"\nSUBMISSION READY: {zip_file} ({zip_file.stat().st_size/1024:.1f} KB)")